# Ingestão da Camada Gold — Star Schema

Notebook para leitura dos dados da camada **Silver** e carga na camada **Gold** do Data Lakehouse.
Na Gold os dados são modelados em **star schema**: tabelas de fato (medições agregadas) e tabelas de dimensão (catálogos descritivos).

## 📚 Sobre este Notebook

Este notebook implementa a camada **Gold** a partir dos dados da camada **Silver**, criando visões agregadas prontas para consumo.

### Arquitetura Medalhão

* **📂 Staging**: Dados brutos do Google Drive (notebook MVP02-Download)
* **🥉 Bronze**: Dados brutos com metadados (notebook MVP03-Bronze)
* **🥈 Silver**: Dados limpos e padronizados (notebook MVP04-Silver)
* **🥇 Gold**: Dados agregados e prontos para consumo (este notebook)

### 🎯 Objetivo
Criar tabelas no formato **star schema** (fatos e dimensões) a partir dos dados da Silver, usando `CREATE OR REPLACE TABLE AS SELECT`:

#### Tabelas de Fato (medições agregadas)
| Tabela Gold | Origem Silver | Agregação |
|-------------|--------------|----------|
| `gold.fato_pluviometria_mensal` | `silver.leitura_pluviometros` | Total e média de chuva por estação/mês |
| `gold.fato_bacias_mensal` | `silver.medicao_bacias` | Média de densidade e totais de água/salmoura por bacia/mês |
| `gold.fato_laje_mensal` | `silver.medicao_laje` | Média das réguas de nível por cristalizador/mês |

#### Tabelas de Dimensão (catálogos descritivos)
| Tabela Gold | Origem | Descrição |
|-------------|--------|-----------|
| `gold.dim_estacao` | `silver.leitura_pluviometros` | Cadastro de estações pluviométricas |
| `gold.dim_bacia` | `silver.medicao_bacias` | Cadastro de bacias evaporadoras |
| `gold.dim_cristalizador` | `silver.medicao_laje` | Cadastro de cristalizadores |
| `gold.dim_tempo` | Derivada das fato_* | Dimensão temporal (ano, mês, trimestre, semestre) |

In [0]:
%sql
-- Define o catálogo ativo como mvpDataEngineer no Unity Catalog.
USE CATALOG mvpDataEngineer;

-- Define o schema ativo como gold dentro do catálogo atual.
USE SCHEMA gold;

## 🥇 Etapa 1: Criação das Tabelas Gold (Star Schema)

Lê os dados da camada **Silver** e cria tabelas no formato **star schema** na camada **Gold** usando `CREATE OR REPLACE TABLE AS SELECT` (CTAS).

### 1a. Tabelas de Fato (medições agregadas)

| Tabela Gold | Métricas calculadas |
|-------------|---------------------|
| `fato_pluviometria_mensal` | Total de chuva (SUM), média diária (AVG), dias medidos (COUNT) por estação/mês |
| `fato_bacias_mensal` | Média de densidade (AVG), totais de água doce e salmoura (SUM) por bacia/mês |
| `fato_laje_mensal` | Média das 8 réguas de nível (AVG) por cristalizador/mês |

### 1b. Tabelas de Dimensão (catálogos descritivos)

| Tabela Gold | Conteúdo |
|-------------|----------|
| `dim_estacao` | Estações pluviométricas com período e total de medições |
| `dim_bacia` | Bacias evaporadoras com período e total de medições |
| `dim_cristalizador` | Cristalizadores com área, período e total de medições |
| `dim_tempo` | Dimensão temporal: ano, mês, nome do mês, trimestre, semestre |

In [0]:
%skip
%sql 

drop table mvpDataEngineer.gold.fato_pluviometria_mensal;
drop table mvpDataEngineer.gold.fato_bacias_mensal;
drop table mvpDataEngineer.gold.fato_laje_mensal;
drop table mvpDataEngineer.gold.dim_estacao;
drop table mvpDataEngineer.gold.dim_bacia;
drop table mvpDataEngineer.gold.dim_tempo;
drop table mvpDataEngineer.gold.dim_cristalizador;

In [0]:
%sql
-- =========================================
-- TABELA DE FATO: fato_pluviometria_mensal
-- =========================================
-- Agrega leituras pluviométricas por estação e mês
-- Métricas: total de chuva, média diária, dias medidos

CREATE OR REPLACE TABLE mvpDataEngineer.gold.fato_pluviometria_mensal AS
SELECT 
  estacao,
  YEAR(data_registro) AS ano,
  MONTH(data_registro) AS mes,
  ROUND(SUM(milimetros), 2) AS total_milimetros,
  ROUND(AVG(milimetros), 2) AS media_diaria,
  COUNT(*) AS dias_medidos,
  current_timestamp() AS data_carga_gold
FROM mvpDataEngineer.silver.leitura_pluviometros
WHERE data_registro IS NOT NULL
GROUP BY estacao, YEAR(data_registro), MONTH(data_registro)
ORDER BY estacao, ano, mes;

In [0]:
%sql
-- =========================================
-- TABELA DE FATO: fato_bacias_mensal
-- =========================================
-- Agrega medições das bacias evaporadoras por bacia e mês
-- Métricas: média de densidade, totais de água doce e salmoura

CREATE OR REPLACE TABLE mvpDataEngineer.gold.fato_bacias_mensal AS
SELECT 
  bacia,
  YEAR(data_registro) AS ano,
  MONTH(data_registro) AS mes,
  ROUND(AVG(salmoura_densidade), 2) AS media_densidade,
  SUM(agua_doce_colocado) AS total_agua_doce_colocado,
  SUM(agua_doce_retirado) AS total_agua_doce_retirado,
  SUM(salmoura_colocado) AS total_salmoura_colocado,
  SUM(salmoura_retirado) AS total_salmoura_retirado,
  COUNT(*) AS registros,
  current_timestamp() AS data_carga_gold
FROM mvpDataEngineer.silver.medicao_bacias
WHERE data_registro IS NOT NULL
GROUP BY bacia, YEAR(data_registro), MONTH(data_registro)
ORDER BY bacia, ano, mes;

In [0]:
%sql
select * FROM mvpDataEngineer.silver.medicao_laje

In [0]:
%sql
-- =========================================
-- TABELA DE FATO: fato_laje_mensal
-- =========================================
-- Agrega medições da laje dos cristalizadores por cristalizador e mês
-- Métricas: média das 8 réguas de nível, área em hectares

CREATE OR REPLACE TABLE mvpDataEngineer.gold.fato_laje_mensal AS
SELECT 
  cristalizador,
  area_hectares,
  YEAR(data_medicao) AS ano,
  MONTH(data_medicao) AS mes,
  ROUND(AVG(regua_1), 1) AS media_regua_1,
  ROUND(AVG(regua_2), 1) AS media_regua_2,
  ROUND(AVG(regua_3), 1) AS media_regua_3,
  ROUND(AVG(regua_4), 1) AS media_regua_4,
  ROUND(AVG(regua_5), 1) AS media_regua_5,
  ROUND(AVG(regua_6), 1) AS media_regua_6,
  ROUND(AVG(regua_7), 1) AS media_regua_7,
  ROUND(AVG(regua_8), 1) AS media_regua_8,
  COUNT(*) AS registros,
  current_timestamp() AS data_carga_gold
FROM mvpDataEngineer.silver.medicao_laje
WHERE data_medicao IS NOT NULL
GROUP BY cristalizador, area_hectares, YEAR(data_medicao), MONTH(data_medicao)
ORDER BY cristalizador, ano, mes;

### 1b. Tabelas de Dimensão (catálogos descritivos)

As tabelas de dimensão armazenam os atributos descritivos que dão contexto às medições das tabelas de fato. São extraídas da camada Silver com `SELECT DISTINCT`.

In [0]:
%sql
-- =========================================
-- TABELA DE DIMENSÃO: dim_estacao
-- =========================================
-- Cadastro de estações pluviométricas com período e total de medições

CREATE OR REPLACE TABLE mvpDataEngineer.gold.dim_estacao AS
SELECT 
  estacao,
  unidade,
  MIN(data_registro) AS primeira_medicao,
  MAX(data_registro) AS ultima_medicao,
  COUNT(*) AS total_medicoes,
  current_timestamp() AS data_carga_gold
FROM mvpDataEngineer.silver.leitura_pluviometros
WHERE data_registro IS NOT NULL
GROUP BY estacao, unidade
ORDER BY estacao;

In [0]:
%sql
-- =========================================
-- TABELA DE DIMENSÃO: dim_bacia
-- =========================================
-- Cadastro de bacias evaporadoras com período e total de medições

CREATE OR REPLACE TABLE mvpDataEngineer.gold.dim_bacia AS
SELECT 
  bacia,
  unidade,
  MIN(data_registro) AS primeira_medicao,
  MAX(data_registro) AS ultima_medicao,
  COUNT(*) AS total_medicoes,
  current_timestamp() AS data_carga_gold
FROM mvpDataEngineer.silver.medicao_bacias
WHERE data_registro IS NOT NULL
GROUP BY bacia, unidade
ORDER BY bacia;

In [0]:
%sql
-- =========================================
-- TABELA DE DIMENSÃO: dim_cristalizador
-- =========================================
-- Cadastro de cristalizadores com área, período e total de medições

CREATE OR REPLACE TABLE mvpDataEngineer.gold.dim_cristalizador AS
SELECT 
  cristalizador,
  area_hectares,
  MIN(data_medicao) AS primeira_medicao,
  MAX(data_medicao) AS ultima_medicao,
  COUNT(*) AS total_medicoes,
  current_timestamp() AS data_carga_gold
FROM mvpDataEngineer.silver.medicao_laje
WHERE data_medicao IS NOT NULL
GROUP BY cristalizador, area_hectares
ORDER BY cristalizador;

In [0]:
%sql
-- =========================================
-- TABELA DE DIMENSÃO: dim_tempo
-- =========================================
-- Dimensão temporal: ano, mês, nome do mês, trimestre e semestre
-- Derivada das combinações de ano/mês presentes nas tabelas de fato

CREATE OR REPLACE TABLE mvpDataEngineer.gold.dim_tempo AS
SELECT 
  ano,
  mes,
  CASE mes
    WHEN 1 THEN 'Janeiro' WHEN 2 THEN 'Fevereiro' WHEN 3 THEN 'Marco'
    WHEN 4 THEN 'Abril' WHEN 5 THEN 'Maio' WHEN 6 THEN 'Junho'
    WHEN 7 THEN 'Julho' WHEN 8 THEN 'Agosto' WHEN 9 THEN 'Setembro'
    WHEN 10 THEN 'Outubro' WHEN 11 THEN 'Novembro' WHEN 12 THEN 'Dezembro'
  END AS nome_mes,
  CONCAT(ano, '-', LPAD(CAST(mes AS STRING), 2, '0')) AS ano_mes,
  CEIL(mes / 3.0) AS trimestre,
  CASE WHEN mes <= 6 THEN 1 ELSE 2 END AS semestre,
  current_timestamp() AS data_carga_gold
FROM (
  SELECT ano, mes FROM mvpDataEngineer.gold.fato_pluviometria_mensal
  UNION
  SELECT ano, mes FROM mvpDataEngineer.gold.fato_bacias_mensal
  UNION
  SELECT ano, mes FROM mvpDataEngineer.gold.fato_laje_mensal
)
ORDER BY ano, mes;

## 🔍 Etapa 2: Validação dos Dados Gold

### O que verificar?
* **Total de registros** agregados por tabela
* **Amostras** dos dados agregados
* **Período coberto** (ano/mês mínimo e máximo)

In [0]:
%sql
-- Resumo das tabelas Gold: total de registros, período coberto
-- Tabelas de Fato
SELECT 
  'FATO_PLUVIOMETRIA_MENSAL' as tabela,
  COUNT(*) as total_registros,
  MIN(CONCAT(ano, '-', mes)) as periodo_min,
  MAX(CONCAT(ano, '-', mes)) as periodo_max
FROM gold.fato_pluviometria_mensal

UNION ALL

SELECT 
  'FATO_BACIAS_MENSAL' as tabela,
  COUNT(*) as total_registros,
  MIN(CONCAT(ano, '-', mes)) as periodo_min,
  MAX(CONCAT(ano, '-', mes)) as periodo_max
FROM gold.fato_bacias_mensal

UNION ALL

SELECT 
  'FATO_LAJE_MENSAL' as tabela,
  COUNT(*) as total_registros,
  MIN(CONCAT(ano, '-', mes)) as periodo_min,
  MAX(CONCAT(ano, '-', mes)) as periodo_max
FROM gold.fato_laje_mensal

UNION ALL
-- Tabelas de Dimensão
SELECT 
  'DIM_ESTACAO' as tabela,
  COUNT(*) as total_registros,
  CAST(MIN(primeira_medicao) AS STRING) as periodo_min,
  CAST(MAX(ultima_medicao) AS STRING) as periodo_max
FROM gold.dim_estacao

UNION ALL

SELECT 
  'DIM_BACIA' as tabela,
  COUNT(*) as total_registros,
  CAST(MIN(primeira_medicao) AS STRING) as periodo_min,
  CAST(MAX(ultima_medicao) AS STRING) as periodo_max
FROM gold.dim_bacia

UNION ALL

SELECT 
  'DIM_CRISTALIZADOR' as tabela,
  COUNT(*) as total_registros,
  CAST(MIN(primeira_medicao) AS STRING) as periodo_min,
  CAST(MAX(ultima_medicao) AS STRING) as periodo_max
FROM gold.dim_cristalizador

UNION ALL

SELECT 
  'DIM_TEMPO' as tabela,
  COUNT(*) as total_registros,
  MIN(ano_mes) as periodo_min,
  MAX(ano_mes) as periodo_max
FROM gold.dim_tempo

ORDER BY tabela;

In [0]:
%sql
-- Amostra de pluviometria_mensal: top 10 estações com maior volume de chuva
SELECT estacao, ano, mes, total_milimetros, media_diaria, dias_medidos
FROM mvpDataEngineer.gold.fato_pluviometria_mensal
ORDER BY total_milimetros DESC
LIMIT 10;

In [0]:
%sql
-- Amostra de bacias_mensal: visão mensal por bacia
SELECT bacia, ano, mes, media_densidade, 
       total_agua_doce_colocado, total_agua_doce_retirado,
       total_salmoura_colocado, total_salmoura_retirado, registros
FROM mvpDataEngineer.gold.fato_bacias_mensal
ORDER BY bacia, ano, mes
--LIMIT 10;

In [0]:
%sql
-- Amostra de laje_mensal: média das réguas por cristalizador/mês
SELECT cristalizador, area_hectares, ano, mes, 
       media_regua_1, media_regua_2, media_regua_3, media_regua_4,
       media_regua_5, media_regua_6, media_regua_7, media_regua_8, registros
FROM mvpDataEngineer.gold.fato_laje_mensal
ORDER BY cristalizador, ano, mes
LIMIT 10;

## 📝 Etapa 3: Documentação das Tabelas e Colunas

Comentários no Unity Catalog tornam os dados autodescritivos, visíveis no Catalog Explorer, autocomplete do SQL Editor e APIs.

In [0]:
%sql
-- =========================================
-- COMENTÁRIOS: fato_pluviometria_mensal
-- =========================================

COMMENT ON TABLE mvpDataEngineer.gold.fato_pluviometria_mensal IS 'Tabela de fato: agregação mensal de leituras pluviométricas por estação (total de chuva, média diária e dias medidos)';

COMMENT ON COLUMN mvpDataEngineer.gold.fato_pluviometria_mensal.estacao IS 'Nome da estação pluviométrica (FK para dim_estacao)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_pluviometria_mensal.ano IS 'Ano da medição (FK para dim_tempo)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_pluviometria_mensal.mes IS 'Mês da medição 1-12 (FK para dim_tempo)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_pluviometria_mensal.total_milimetros IS 'Soma total de chuva no mês em milímetros (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_pluviometria_mensal.media_diaria IS 'Média diária de chuva no mês em milímetros (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_pluviometria_mensal.dias_medidos IS 'Número de dias com medição no mês';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_pluviometria_mensal.data_carga_gold IS 'Timestamp da carga na camada Gold';

-- =========================================
-- COMENTÁRIOS: fato_bacias_mensal
-- =========================================

COMMENT ON TABLE mvpDataEngineer.gold.fato_bacias_mensal IS 'Tabela de fato: agregação mensal das bacias evaporadoras (média de densidade e totais de água/salmoura)';

COMMENT ON COLUMN mvpDataEngineer.gold.fato_bacias_mensal.bacia IS 'Identificador da bacia evaporadora (FK para dim_bacia)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_bacias_mensal.ano IS 'Ano da medição (FK para dim_tempo)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_bacias_mensal.mes IS 'Mês da medição 1-12 (FK para dim_tempo)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_bacias_mensal.media_densidade IS 'Média da densidade da salmoura no mês em g/cm³ (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_bacias_mensal.total_agua_doce_colocado IS 'Total de água doce colocada no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_bacias_mensal.total_agua_doce_retirado IS 'Total de água doce retirada no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_bacias_mensal.total_salmoura_colocado IS 'Total de salmoura colocada no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_bacias_mensal.total_salmoura_retirado IS 'Total de salmoura retirada no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_bacias_mensal.registros IS 'Número de medições no mês';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_bacias_mensal.data_carga_gold IS 'Timestamp da carga na camada Gold';

-- =========================================
-- COMENTÁRIOS: fato_laje_mensal
-- =========================================

COMMENT ON TABLE mvpDataEngineer.gold.fato_laje_mensal IS 'Tabela de fato: agregação mensal da laje dos cristalizadores (média das 8 réguas de nível)';

COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.cristalizador IS 'Número de identificação do cristalizador (FK para dim_cristalizador)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.area_hectares IS 'Área do cristalizador em hectares (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.ano IS 'Ano da medição (FK para dim_tempo)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.mes IS 'Mês da medição 1-12 (FK para dim_tempo)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.media_regua_1 IS 'Média da leitura da régua 1 no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.media_regua_2 IS 'Média da leitura da régua 2 no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.media_regua_3 IS 'Média da leitura da régua 3 no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.media_regua_4 IS 'Média da leitura da régua 4 no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.media_regua_5 IS 'Média da leitura da régua 5 no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.media_regua_6 IS 'Média da leitura da régua 6 no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.media_regua_7 IS 'Média da leitura da régua 7 no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.media_regua_8 IS 'Média da leitura da régua 8 no mês (INT)';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.registros IS 'Número de medições no mês';
COMMENT ON COLUMN mvpDataEngineer.gold.fato_laje_mensal.data_carga_gold IS 'Timestamp da carga na camada Gold';

-- =========================================
-- COMENTÁRIOS: dim_estacao
-- =========================================

COMMENT ON TABLE mvpDataEngineer.gold.dim_estacao IS 'Tabela de dimensão: cadastro de estações pluviométricas com período e total de medições';

COMMENT ON COLUMN mvpDataEngineer.gold.dim_estacao.estacao IS 'Nome da estação pluviométrica (PK)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_estacao.unidade IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_estacao.primeira_medicao IS 'Data da primeira medição registrada (DATE)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_estacao.ultima_medicao IS 'Data da última medição registrada (DATE)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_estacao.total_medicoes IS 'Número total de medições da estação';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_estacao.data_carga_gold IS 'Timestamp da carga na camada Gold';

-- =========================================
-- COMENTÁRIOS: dim_bacia
-- =========================================

COMMENT ON TABLE mvpDataEngineer.gold.dim_bacia IS 'Tabela de dimensão: cadastro de bacias evaporadoras com período e total de medições';

COMMENT ON COLUMN mvpDataEngineer.gold.dim_bacia.bacia IS 'Identificador da bacia evaporadora (PK)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_bacia.unidade IS 'Unidade federativa de origem (ex: RN)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_bacia.primeira_medicao IS 'Data da primeira medição registrada (DATE)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_bacia.ultima_medicao IS 'Data da última medição registrada (DATE)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_bacia.total_medicoes IS 'Número total de medições da bacia';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_bacia.data_carga_gold IS 'Timestamp da carga na camada Gold';

-- =========================================
-- COMENTÁRIOS: dim_cristalizador
-- =========================================

COMMENT ON TABLE mvpDataEngineer.gold.dim_cristalizador IS 'Tabela de dimensão: cadastro de cristalizadores com área, período e total de medições';

COMMENT ON COLUMN mvpDataEngineer.gold.dim_cristalizador.cristalizador IS 'Número de identificação do cristalizador (PK)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_cristalizador.area_hectares IS 'Área do cristalizador em hectares (DOUBLE)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_cristalizador.primeira_medicao IS 'Data da primeira medição registrada (DATE)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_cristalizador.ultima_medicao IS 'Data da última medição registrada (DATE)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_cristalizador.total_medicoes IS 'Número total de medições do cristalizador';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_cristalizador.data_carga_gold IS 'Timestamp da carga na camada Gold';

-- =========================================
-- COMENTÁRIOS: dim_tempo
-- =========================================

COMMENT ON TABLE mvpDataEngineer.gold.dim_tempo IS 'Tabela de dimensão: dimensão temporal com ano, mês, nome do mês, trimestre e semestre';

COMMENT ON COLUMN mvpDataEngineer.gold.dim_tempo.ano IS 'Ano (PK)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_tempo.mes IS 'Mês 1-12 (PK)';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_tempo.nome_mes IS 'Nome do mês em português';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_tempo.ano_mes IS 'Combinação ano-mês no formato YYYY-MM';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_tempo.trimestre IS 'Trimestre 1-4';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_tempo.semestre IS 'Semestre 1-2';
COMMENT ON COLUMN mvpDataEngineer.gold.dim_tempo.data_carga_gold IS 'Timestamp da carga na camada Gold';